# 01 - Preparação de Dados

Carrega o arquivo `espelhos_acordaos_artigo2026.parquet` e o complementa com:
1. **Inteiro Teor** de cada acórdão, obtido dos ZIPs publicados na Base de Dados Abertos do STJ (dataset `integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica`).
2. **Dados do espelho** de cada acórdão (tese jurídica, tema, referências legislativas, etc.), obtidos dos JSONs de espelhos publicados no Portal de Dados Abertos do STJ.

- Dados Abertos: https://dadosabertos.web.stj.jus.br/group/jurisprudencia
- Sobre o espelho do acórdão: https://scon.stj.jus.br/docs_internet/jurisprudencia/ajuda/Espelho_do_Acordao_atualizado.pdf
- Sobre a api CKAN: https://docs.ckan.org/en/2.9/api/

Textos:
- No Portal de Dados Abertos, o dataset de espelhos contém a Ementa do Acórdão, enquanto o dataset de integras contém o Inteiro Teor.
- Outros campos textuais do espelho não estão completos ou não estão no padrão que é o objetivo do experimento, por isso não podem der usados como tabela da verdade no experimento

No caso de haver problemas com o download de algum arquivo usando a api, pode-se baixar manualmente o arquivo e colocar na pasta de cache:
- Download manual dos arquivos com as íntegras: https://dadosabertos.web.stj.jus.br/dataset/integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica
- Download manual dos arquivos com os espelhos: https://dadosabertos.web.stj.jus.br/organization/superior-tribunal-de-justica?tags=jurisprudencia

## Pipeline
```
parquet → set(seq_documento_acordao) + set(num_registro)
         ↓
CKAN → lista de ZIPs (íntegras) + lista de JSONs (espelhos por órgão)
         ↓  (download com cache)
ZIP em memória → filtro por seq_documento → coluna 'texto'
JSON de espelho → filtro por num_registro  → colunas 'teseJuridica', 'tema', etc.
         ↓
parquet enriquecido com texto + campos do espelho
```

## 1. Configuração e Imports

In [1]:
# Importação do pacote utilitário
import os, sys
from pathlib import Path
try:
   sys.path.extend(['../','../src'])
   from util import UtilEnv
   if UtilEnv.carregar_env(pastas=['./','../']):
      print('✅ Variáveis de ambiente carregadas _o/')
except:
    UtilEnv = None
    print('⚠️ Não foi possível importar o pacote para carregar o .env!')


# ─── Arquivos ─────────────────────────────────────────────────────────────────
PASTA_DATA = '../data'
PARQUET_BASE  = os.path.join(PASTA_DATA, 'espelhos_acordaos_artigo2026.parquet')
PARQUET_SAIDA = os.path.join(PASTA_DATA, 'espelhos_acordaos_artigo2026_com_texto.parquet')

# Pasta raiz para cache dos ZIPs e JSONs (evita re-download)
# As subpastas espelhos/, integras/ e metadados_integras/ são criadas automaticamente pela UtilCkan
# Padrão ./downloads_stj
# DOWNLOAD_DIR = Path('downloads_stj')  

# Define se tenta baixar os zips das íntegras e dos espelhos ou apenas consolida o que já existir
CKAN_TIMEOUT = UtilEnv.get_int('CKAN_TIMEOUT',300) if UtilEnv else 600
ATUALIZAR_CACHE_E_MAPAS = UtilEnv.env_true('ATUALIZAR_CACHE_E_MAPAS',True) if UtilEnv else True

if UtilEnv is None:
   print('⚠️ Variáveis de ambiente não carregadas, verfique pacote Util emc ./src!')
print('Configurações:',
      f'CKAN_TIMEOUT: {CKAN_TIMEOUT}',
      f'ATUALIZAR_CACHE_E_MAPAS: {ATUALIZAR_CACHE_E_MAPAS}',
      f'',
      sep='\n')


Env encontrado em ../.env
✅ Variáveis de ambiente carregadas _o/
Configurações:
CKAN_TIMEOUT: 600
ATUALIZAR_CACHE_E_MAPAS: False



## 2. Carregamento do Parquet Base

In [2]:
import pandas as pd
df = pd.read_parquet(PARQUET_BASE)

# Cria id_peca calculado: número de registro . data de publicação . 
# Exemplo: '202202853462.20230601.'
# Esses serão os nomes dos arquivos de extração do experimento
df['id_peca'] = (
    df['registro'].astype(str)
    + '.'
    + df['publicacao'].astype(str).str.replace('-','')
    + '.'
)

print(f'Total de registros : {len(df)}')
print(f'Colunas            : {df.columns.tolist()}')
print(f'Exemplo id_peca    : {df["id_peca"].iloc[0]}')
print()
df.head(3)


Total de registros : 1225
Colunas            : ['ano', 'documento', 'registro', 'publicacao', 'id_peca']
Exemplo id_peca    : 202202853462.20230601.



,ano,documento,registro,publicacao,id_peca
0,2023,188798478,202202853462,2023-06-01,202202853462.20230601.
1,2022,156649445,202201555326,2022-06-20,202201555326.20220620.
2,2022,155998284,202201341093,2022-06-15,202201341093.20220615.


In [3]:
# prepara os filtros para uso de UtilCKan
documentos_necessarios = None
registros_necessarios = None

if 'acordao' in df.columns:
    # ── Lookup: seq_documento_acordao → índices do dataframe (para junção de textos) ──
    seq_para_idx: dict[int, list[int]] = {}
    for idx, val in df['acordao'].dropna().items():
        seq = int(val)
        seq_para_idx.setdefault(seq, []).append(idx)
    documentos_necessarios = set(seq_para_idx.keys())
    print(f'seq_documento_acordao únicos necessários: {len(documentos_necessarios)}')
    print(f'Exemplos: {list(documentos_necessarios)[:5]}')

elif 'registro' in df.columns and 'publicacao' in df.columns:
    # ── Lookup por numero_registro + data_publicacao ──
    if 'tipo_decisao' in df.columns:
        registros_necessarios = set(zip(df['registro'].astype(str), df['publicacao'].astype(str), df['tipo_decisao'].astype(str)))
    else:
        registros_necessarios = set(zip(df['registro'].astype(str), df['publicacao'].astype(str)))
    print(f'registros_necessarios únicos: {len(registros_necessarios)}')
    print(f'Exemplos: {list(registros_necessarios)[:5]}')

elif 'registro' in df.columns:
    registros_necessarios = set(df['registro'].dropna().astype(str).unique())
    print(f'registros_necessarios únicos: {len(registros_necessarios)}')
    print(f'Exemplos: {list(registros_necessarios)[:5]}')


# ── Anos presentes no parquet — filtro de ZIPs por prefixo de nome ──
anos_necessarios = set()
if 'ano' in df.columns:
    anos_necessarios = {str(int(a)) for a in df['ano'].dropna().unique()}
elif 'publicacao' in df.columns:
    # Extrai do YYYY-MM-DD
    anos_necessarios = {str(d)[:4] for d in df['publicacao'].dropna().unique()}

print(f'\nAnos necessários: {sorted(anos_necessarios)}\n')


registros_necessarios únicos: 1225
Exemplos: [('202300454280', '2023-05-26'), ('202300031337', '2023-08-17'), ('202203287921', '2022-12-02'), ('202300296930', '2023-05-11'), ('202301253551', '2023-08-30')]

Anos necessários: ['2022', '2023', '2024', '2025']



## 2. Extração via UtilCkan
Instanciação da classe com os filtros exatos de documentos e registros.

In [4]:
# ── Configuração do utilitário com os filtros ──
from util_ckan import UtilCkan
ckan = UtilCkan(
    anos              = anos_necessarios if anos_necessarios else None,
    documentos        = documentos_necessarios,
    registros         = registros_necessarios,
    timeout           = CKAN_TIMEOUT,
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,
)

print(f'CKAN        : {ckan.base_url}')
print(f'Parquet     : {PARQUET_BASE}')
print(f'Cache dir   : {ckan.download_dir}')

# ── 1. Processamento de Espelhos e Íntegras ──
print('▶️  Iniciando cruzamento e carga através da api gerar_dataset_espelhos()...')
df_ckan = ckan.gerar_dataset_espelhos(
    caminho_saida = PARQUET_SAIDA.replace('.parquet', '_temp_ckan.parquet'),
    incluir_integras = True,
    incluir_ementas = True,
    incluir_decisoes = True
)
if df_ckan is None:
    print("Nenhum dado retornado do CKAN.")
    df_ckan = pd.DataFrame()
else:
    print(f"Colunas extraídas do CKAN: {df_ckan.columns.tolist()}")


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
CKAN        : https://dadosabertos.web.stj.jus.br
Parquet     : ../data/espelhos_acordaos_artigo2026.parquet
Cache dir   : downloads_stj
▶️  Iniciando cruzamento e carga através da api gerar_dataset_espelhos()...
Registros cruzados (espelho × íntegra): 1225


Espelhos:   0%|          | 0/52 [00:00<?, ?it/s]


Registros extraídos: 1226


Extraindo íntegras:   0%|          | 0/131 [00:00<?, ?it/s]

Íntegras extraídas: 1072 / 1072
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_artigo2026_com_texto_temp_ckan.parquet
      10.12 MB | 25 colunas | 1225 registros
───────────────────────────────────────────────────────
  Com íntegra  :   1072  (87.5%)
  Sem íntegra  :    153  (12.5%)
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2022 →    38 registros | íntegra: 9
      2023 →   924 registros | íntegra: 800
      2024 →   258 registros | íntegra: 258
      2025 →     5 registros | íntegra: 5
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      AgRg no HC                            856
      AgRg no AREsp                         208
      HC                                     67
      AgRg nos EDcl no HC                    18
      EDcl no HC                             14
      EDcl no AgRg no HC                     12
      EDcl no AgRg no AREsp                  11
      RC

## 3. Junção e Salvamento
Realiza o merge dos novos textos no dataframe original e salva o parquet.

In [5]:
# ── 2. Junção com DataFrame Original ──
df_resultado = df.copy()

if not df_ckan.empty:
    # Escolha da chave para o merge baseado no DataFrame
    key_df = None
    key_ckan = None
    
    if 'acordao' in df_resultado.columns and 'seq_documento_acordao' in df_ckan.columns:
        key_df = 'acordao'
        key_ckan = 'seq_documento_acordao'
    elif 'registro' in df_resultado.columns and 'numeroRegistro' in df_ckan.columns:
        key_df = 'registro'
        key_ckan = 'numeroRegistro'

    if key_df and key_ckan:
        print(f"Merge usando chave: df_resultado['{key_df}'] == df_ckan['{key_ckan}']")
        
        # Garante o mesmo formato
        df_ckan[key_ckan] = df_ckan[key_ckan].astype(str)
        df_resultado[key_df] = df_resultado[key_df].astype(str)
        
        # Remove colunas que já existem para não duplicar, exceto a chave
        cols_novas = [c for c in df_ckan.columns if c not in df_resultado.columns and c != key_ckan]
        df_ckan_dedup = df_ckan[[*cols_novas, key_ckan]].copy()
        
        df_resultado = df_resultado.merge(
            df_ckan_dedup, 
            left_on=key_df, 
            right_on=key_ckan, 
            how='left'
        )
        if key_ckan in df_resultado.columns and key_ckan != key_df:
            df_resultado.drop(columns=[key_ckan], inplace=True)
            
        print(f'Colunas do CKAN adicionadas: {cols_novas}')
    else:
        print(f"⚠️  Não foi possível identificar coluna chave compatível. Verifique as colunas de df e df_ckan.")
else:
    print("DataFrame do CKAN vazio, pulando merge.")

df_resultado['integra'] = df_resultado['integra'].fillna('') if 'integra' in df_resultado.columns else ''

# ── Estatísticas da extração ──────────────────────────────────────────────────
# id_mapa é adicionado pelo CKAN para todo registro com espelho correspondente
total        = len(df_resultado)
com_integra  = df_resultado['integra'].str.len().gt(0).sum() if 'integra' in df_resultado.columns else 0
sem_integra  = total - com_integra
com_espelho  = df_resultado['id_mapa'].notna().sum() if 'id_mapa' in df_resultado.columns else 0
dups_filtrados = ckan.obter_duplicados(df_resultado)
n_dups_ids     = len(dups_filtrados)
n_dups_total   = sum(len(v) for v in dups_filtrados.values())

print()
print('=== RESULTADO ===')
print(f'Total de registros : {total}')
print(f'Com texto          : {com_integra}')
print(f'Sem texto          : {sem_integra}')
print(f'Com espelho        : {com_espelho}')
print(f'IDs com duplicatas : {n_dups_ids}  ({n_dups_total} ocorrência(s) duplicada(s))')


Merge usando chave: df_resultado['registro'] == df_ckan['numeroRegistro']
Colunas do CKAN adicionadas: ['id', 'siglaClasse', 'descricaoClasse', 'nomeOrgaoJulgador', 'ministroRelator', 'tipoDeDecisao', 'dataPublicacao', 'dataDecisao', 'teseJuridica', 'tema', 'referenciasLegislativas', 'jurisprudenciaCitada', 'notas', 'termosAuxiliares', 'informacoesComplementares', 'acordaosSimilares', 'ementa', 'decisao', 'id_mapa', 'orgao', 'data_publicacao_iso', 'seq_documento_acordao', 'tem_integra', 'integra']

=== RESULTADO ===
Total de registros : 1243
Com texto          : 1089
Sem texto          : 154
Com espelho        : 1243
IDs com duplicatas : 22  (33 ocorrência(s) duplicada(s))


---

In [6]:
# Exibe exemplos de registros com texto e campos de espelho extraídos
if 'integra' in df_resultado.columns:
    com_integra_df = df_resultado[df_resultado['integra'].str.len() > 0]
    
    if not com_integra_df.empty:
        for i, row in com_integra_df[:2].iterrows():
            dados = [f'{str(c).ljust(25)}: {v}' for c, v in row.items() if c != 'integra' and v is not None ]
            dados.sort()
            [print(_) for _ in dados]
            # Exibir resumo do conteúdo 
            _texto = f'{row["integra"][:100]}\n[..]{row["integra"][-100:]}'.replace('\r','\n')
            print(f'\nÍNTEGRA: {_texto}')
            print('-' * 60)
    else:
        print('⚠️  Nenhuma íntegra encontrado ainda. Verifique os ZIPs disponíveis no CKAN.')
else:
    print('⚠️  Coluna integra não foi associada ao df_resultado.')


acordaosSimilares        : []
ano                      : 2023
dataDecisao              : 20230510
dataPublicacao           : DJE        DATA:01/06/2023
data_publicacao_iso      : 20230601
decisao                  : Vistos e relatados estes a utos em que são partes as acima indicadas, acordam os Ministros da Terceira Seção, por unanimidade, conceder a ordem para absolver o Paciente da imputação que lhe foi dirigida na Ação Penal n. 013373-74.2020.8.19.0008, com fundamento no art. 386, inciso VII, do Código de Processo Penal e conceder ordem, de ofício, para determinar sua soltura imediata em relação a todos os processos, cabendo aos Juízos e Tribunais, nas ações em curso, e aos Juízos da Execução Penal, nas ações transitadas em julgado, aferirem se a dinâmica probatória é exatamente a mesma repelida nestes autos, e determinar a expedição de ofício comunicando a íntegra desse julgado à Corregedoria de Polícia do Estado do Rio de Janeiro para apuração de eventuais responsabilidades, nos t

## 4. Métricas Finais

In [7]:
df_resultado.to_parquet(PARQUET_SAIDA, index=False)

# ── Métricas básicas ──────────────────────────────────────────────────────────
total        = len(df_resultado)
com_integra  = df_resultado['integra'].str.len().gt(0).sum() if 'integra' in df_resultado.columns else 0
sem_integra  = total - com_integra
# id_mapa é adicionado pelo CKAN para todo registro com espelho correspondente
com_espelho  = df_resultado['id_mapa'].notna().sum() if 'id_mapa' in df_resultado.columns else 0
sem_espelho  = total - com_espelho
# Duplicatas apenas dos id_mapa presentes no resultado filtrado
dups_filtrados = ckan.obter_duplicados(df_resultado)
n_dups_ids     = len(dups_filtrados)
n_dups_total   = sum(len(v) for v in dups_filtrados.values())

tamanhos     = df_resultado.loc[df_resultado['integra'].str.len() > 0, 'integra'].str.len() if 'integra' in df_resultado.columns else pd.Series(dtype=int)
int_media    = tamanhos.mean()    if not tamanhos.empty else 0
int_mediana  = tamanhos.median()  if not tamanhos.empty else 0
int_min      = tamanhos.min()     if not tamanhos.empty else 0
int_max      = tamanhos.max()     if not tamanhos.empty else 0

sep = '─' * 55
print(sep)
print('  ✅  ARQUIVO GERADO')
print(sep)
print(f'  Arquivo   : {PARQUET_SAIDA}')
print(f'  Tamanho   : {Path(PARQUET_SAIDA).stat().st_size / 1024**2:.2f} MB')
print(f'  Colunas   : {len(df_resultado.columns)}  →  {df_resultado.columns.tolist()}')

print(sep)
print('  📊  COBERTURA DOS DADOS')
print(sep)
print(f'  Total de registros        : {total:>6}')
print(f'  Com texto integral        : {com_integra:>6}  ({com_integra/total*100:.1f}%)')
print(f'  Sem texto integral        : {sem_integra:>6}  ({sem_integra/total*100:.1f}%)')
print(f'  Com dados de espelho      : {com_espelho:>6}  ({com_espelho/total*100:.1f}%)')
print(f'  Sem dados de espelho      : {sem_espelho:>6}  ({sem_espelho/total*100:.1f}%)')
if n_dups_ids:
    print(f'  ⚠️  IDs com duplicatas    : {n_dups_ids:>6}  ({n_dups_total} ocorrência(s) — use ckan.obter_duplicados())')

print(sep)
print('  📏  TAMANHO DOS TEXTOS INTEGRAIS (chars)')
print(sep)
if com_integra > 0:
    print(f'  Média                     : {int_media:>10.0f}')
    print(f'  Mediana                   : {int_mediana:>10.0f}')
    print(f'  Mínimo                    : {int_min:>10.0f}')
    print(f'  Máximo                    : {int_max:>10.0f}')
    print(f'  Total de caracteres       : {tamanhos.sum():>10,.0f}')
else:
    print('  (nenhum texto disponível)')

if 'ano' in df_resultado.columns:
    print(sep)
    print('  📅  REGISTROS POR ANO')
    print(sep)
    por_ano = df_resultado.groupby('ano').agg(
        total     = ('ano', 'count'),
        com_integra = ('integra', lambda x: x.str.len().gt(0).sum()) if 'integra' in df_resultado.columns else ('ano', lambda x: 0),
    ).sort_index()
    for ano, row in por_ano.iterrows():
        pct = row['com_integra'] / row['total'] * 100 if row['total'] else 0
        print(f'  {int(ano)}  →  {row["total"]:>5} registros | texto: {row["com_integra"]:>5} ({pct:.0f}%)')

if 'nomeOrgaoJulgador' in df_resultado.columns:
    print(sep)
    print('  ⚖️   REGISTROS POR ÓRGÃO JULGADOR (top 10)')
    print(sep)
    por_orgao = (
        df_resultado['nomeOrgaoJulgador']
        .fillna('(não informado)')
        .value_counts()
        .head(10)
    )
    for orgao, cnt in por_orgao.items():
        print(f'  {str(orgao):<35} : {cnt:>5}')

print(sep)
print('  ✅  Processamento concluído!')
print(sep)


───────────────────────────────────────────────────────
  ✅  ARQUIVO GERADO
───────────────────────────────────────────────────────
  Arquivo   : ../data/espelhos_acordaos_artigo2026_com_texto.parquet
  Tamanho   : 10.16 MB
  Colunas   : 29  →  ['ano', 'documento', 'registro', 'publicacao', 'id_peca', 'id', 'siglaClasse', 'descricaoClasse', 'nomeOrgaoJulgador', 'ministroRelator', 'tipoDeDecisao', 'dataPublicacao', 'dataDecisao', 'teseJuridica', 'tema', 'referenciasLegislativas', 'jurisprudenciaCitada', 'notas', 'termosAuxiliares', 'informacoesComplementares', 'acordaosSimilares', 'ementa', 'decisao', 'id_mapa', 'orgao', 'data_publicacao_iso', 'seq_documento_acordao', 'tem_integra', 'integra']
───────────────────────────────────────────────────────
  📊  COBERTURA DOS DADOS
───────────────────────────────────────────────────────
  Total de registros        :   1243
  Com texto integral        :   1089  (87.6%)
  Sem texto integral        :    154  (12.4%)
  Com dados de espelho      :   